# 01 — Exploración Inicial del Dataset
**Proyecto:** Sistema de Clasificación de Riesgo Académico Estudiantil  
**Dataset:** Students Performance Dataset — Rabie El Kharoua (Kaggle, 2024)  
**Fecha:** Junio 2026

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette('Blues_d')
pd.set_option('display.max_columns', None)

## 1. Carga del dataset

In [ ]:
df = pd.read_csv('../data/raw/students_performance.csv')
print('Dataset cargado exitosamente')
print(f'Dimensiones: {df.shape[0]} filas x {df.shape[1]} columnas')

## 2. Estructura del dataset

In [ ]:
# Tipos de variables
print('Tipos de datos por columna:')
print(df.dtypes)
print('\n')
print('Información general:')
df.info()

In [ ]:
# Primeras filas
print('Primeras 5 filas del dataset:')
df.head()

## 3. Estadísticas descriptivas

In [ ]:
# Resumen estadístico
print('Resumen estadístico de variables numéricas:')
df.describe().round(2)

## 4. Calidad de los datos

In [ ]:
# Valores nulos
nulos = df.isnull().sum()
print('Valores nulos por columna:')
print(nulos)
print(f'\nTotal de valores nulos: {nulos.sum()}')

In [ ]:
# Duplicados
duplicados = df.duplicated().sum()
print(f'Registros duplicados: {duplicados}')

## 5. Distribución de la variable objetivo

In [ ]:
# Distribución de GradeClass
print('Distribución de la variable objetivo GradeClass:')
dist = df['GradeClass'].value_counts().sort_index()
labels = {0: 'A (Excelente)', 1: 'B (Bueno)', 2: 'C (Promedio)', 3: 'D (Bajo)', 4: 'F (Reprobado)'}
for k, v in dist.items():
    print(f'  Clase {k} — {labels[k]}: {v} estudiantes ({v/len(df)*100:.1f}%)')

In [ ]:
# Visualización de la distribución
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico de barras
colores = ['#1a5276', '#2874a6', '#5dade2', '#aed6f1', '#d6eaf8']
etiquetas = ['A\n(Excelente)', 'B\n(Bueno)', 'C\n(Promedio)', 'D\n(Bajo)', 'F\n(Reprobado)']
valores = [df['GradeClass'].value_counts().sort_index()[i] for i in range(5)]

axes[0].bar(etiquetas, valores, color=colores, edgecolor='white', linewidth=1.5)
axes[0].set_title('Distribución de la Variable Objetivo (GradeClass)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Categoría de Rendimiento', fontsize=11)
axes[0].set_ylabel('Número de Estudiantes', fontsize=11)
for i, v in enumerate(valores):
    axes[0].text(i, v + 10, str(v), ha='center', fontweight='bold')

# Gráfico de torta
axes[1].pie(valores, labels=etiquetas, colors=colores, autopct='%1.1f%%', startangle=90)
axes[1].set_title('Proporción por Categoría', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('../docs/distribucion_target.png', dpi=150, bbox_inches='tight')
plt.show()
print('Gráfico guardado en docs/distribucion_target.png')

## 6. Análisis de variables de entrada clave

In [ ]:
# Correlación entre variables numéricas y GradeClass
vars_numericas = ['StudyTimeWeekly', 'Absences', 'GPA', 'Age']
print('Correlación con la variable objetivo GradeClass:')
for var in vars_numericas:
    corr = df[var].corr(df['GradeClass'])
    print(f'  {var}: {corr:.3f}')

In [ ]:
# Distribución de horas de estudio por categoría de rendimiento
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# StudyTimeWeekly vs GradeClass
df.groupby('GradeClass')['StudyTimeWeekly'].mean().plot(
    kind='bar', ax=axes[0], color=colores, edgecolor='white'
)
axes[0].set_title('Horas de Estudio Promedio por Categoría', fontsize=12, fontweight='bold')
axes[0].set_xlabel('GradeClass (0=A ... 4=F)', fontsize=10)
axes[0].set_ylabel('Horas de estudio semanales promedio', fontsize=10)
axes[0].set_xticklabels(['A', 'B', 'C', 'D', 'F'], rotation=0)

# Absences vs GradeClass
df.groupby('GradeClass')['Absences'].mean().plot(
    kind='bar', ax=axes[1], color=colores, edgecolor='white'
)
axes[1].set_title('Ausencias Promedio por Categoría', fontsize=12, fontweight='bold')
axes[1].set_xlabel('GradeClass (0=A ... 4=F)', fontsize=10)
axes[1].set_ylabel('Número de ausencias promedio', fontsize=10)
axes[1].set_xticklabels(['A', 'B', 'C', 'D', 'F'], rotation=0)

plt.tight_layout()
plt.savefig('../docs/variables_vs_target.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Conclusiones de la exploración

- El dataset cuenta con **2.392 registros y 15 columnas**, sin valores nulos ni duplicados.
- La variable objetivo `GradeClass` presenta un **desbalance moderado**: la clase C (promedio) es la más frecuente, mientras que F (reprobado) es la menos frecuente. Esto justifica el uso de **F1-score macro** como métrica.
- `StudyTimeWeekly` y `Absences` muestran correlaciones claras con el rendimiento: a mayor tiempo de estudio, mejor categoría; a mayor ausentismo, peor categoría.
- `GPA` es la variable con mayor correlación directa con `GradeClass`, aunque debe evaluarse el riesgo de *data leakage*.
- El dataset es pertinente y de alta calidad para la tarea de clasificación propuesta.